In [19]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [21]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:

def llm1(prompt1):
#this is for open api
#    response = openai_client.responses.create(
 #   model="gpt-5.4-mini",
  #  input=prompt
#) 
  
    response = openai_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": prompt1}
        ]
    )
    print(response.usage)
    return response.choices[0].message.content
    


'def llm(prompt):\n    response = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=prompt\n) this is for openai\n    response = openai_client.chat.completions.create(\n        model="openai/gpt-oss-20b",\n        messages=[\n            {"role": "user", "content": prompt}\n        ]\n    )\n    print(response.usage)\n    return response.choices[0].message.content\n    '

In [4]:
import os

key = os.getenv("GROQ_API_KEY")
print(key[:12] if key else None)

gsk_hAlS4LZN


In [ ]:
llm1("so i just discovered a course, can you explain")

'Sure thing! What course did you find? Let me know the name (or a link if you have one) and a bit about what it covers, and I can give you an overview, explain its key concepts, and even highlight how it might fit into your learning goals.'

In [ ]:

context1 = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""


In [ ]:
'''question = "I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?"'''
question1 = "fee details of this course?"

prompt1 = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question1}

Context:
{context1}
"""

In [ ]:
answer1 = llm1(prompt1)
print(answer1)

I don't know.


In [22]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [23]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [25]:
documents[0]

{'id': '4487db3924',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I access the course modules and materials?',
 'answer': "All course materials are in the GitHub repo. Each module has its own folder (e.g. `01-overview`, `03-mcp`), and cohort-specific homework is under the `cohorts/` folder.\n\nFor the 2026 cohort, the week-by-week folders are:\n\n- Week 1: [01-ai-native-workflow](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/01-ai-native-workflow)\n- Week 2: [02-development](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/02-development)\n- Week 3: [03-deployment](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/03-deployment)\n- Week 4: [04-devops](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/04-devops)\n\nStart from the folder for your current week — it gathers everything for that week in one place.\n\n

In [26]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
question2 = "I just discovered the course. Can I join now?"

search_results1 = index.search(
    question2,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results1

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [ ]:
def search(question, course="mlops-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

#search_results = search(question)
#print(search_results)

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""


USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()
#print(build_context(search_results))


General Course-Related Questions
Q: Course - Can I still join the course after the start date?
A: Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.

Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.

General Course-Related Questions
Q: Homework: Just found this course, can I still submit homeworks?
A: To clarify on **late homework submissions**:

- You cannot submit after the homework is scored, as the form is closed.
- Once the form is closed (i.e., scored), no further submissions are possible.
- You can check your code against the solution by reviewing the `homework.md` file.

If the due date has passed but the form is still "Open/Submittable":

- This is considered a "late homework submission," and the form is still editable.
- Don’t forget to click the Update button to save any changes.

Please note, it's uncertain 

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

#user_prompt = build_prompt(question, search_results)

#user_prompt(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: Course - Can I still join the course after the start date?
A: Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.

Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.

General Course-Related Questions
Q: Homework: Just found this course, can I still submit homeworks?
A: To clarify on **late homework submissions**:

- You cannot submit after the homework is scored, as the form is closed.
- Once the form is closed (i.e., scored), no further submissions are possible.
- You can check your code against the solution by reviewing the `homework.md` file.

If the due date has passed but the form is still "Open/Submittable":

- This is considered a "late homework submission," and the form is still editable.
- Don’t forget to click th

In [38]:
def llm(INSTRUCTIONS,user_prompt, model):
    message_history = [
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": user_prompt}
    ]
    '''response = openai_client.responses.create(
    model=model,
    input=message_history
)'''
    response = openai_client.chat.completions.create(
        model=model,
        messages= message_history
    )
    print(response.usage)
    return response.choices[0].message.content

In [39]:
def rag(query, model="openai/gpt-oss-20b"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model)
    return answer

final_answer = rag("How do I get a certificate")
print(final_answer)


CompletionUsage(completion_tokens=185, prompt_tokens=751, total_tokens=936, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=46, rejected_prediction_tokens=None, text_tokens=None), prompt_tokens_details=None, queue_time=0.40501363, prompt_time=0.037541876, completion_time=0.243951274, total_time=0.28149315)
To receive a certificate for this course, you must:

1. **Be enrolled in a live cohort** – certificates are only awarded for students in a live cohort.  
2. **Complete the Capstone project** – finish the project and submit it while a live cohort is accepting submissions.  
3. **Fulfill the required peer reviews** – submit and receive the necessary peer reviews during the live cohort period.

Homework is not required for certification. You can work through the material on your own time, but the capstone project and peer reviews must occur while the live cohort is active. Once you meet those criteria, you’ll receive